In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import holidays
import json
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt

df_visitas = pd.read_csv('../data/dataset_global_raw.csv')
df_visitas['fecha'] = pd.to_datetime(df_visitas['fecha'])

df_clima = pd.read_csv('../data/raw/clima_cache.csv')
df_clima['fecha'] = pd.to_datetime(df_clima['fecha'])

with open('../todas_las_ubicaciones.json', 'r', encoding='utf-8') as f:
    datos_loc = json.load(f)

mapa_regiones = {}
mapa_nombres = {}
for org in datos_loc:
    for loc in org.get('locations', []):
        mapa_nombres[loc['uuid']] = loc['name']
        if 'region_code' in loc:
            mapa_regiones[loc['uuid']] = loc['region_code']

df_visitas['region_code'] = df_visitas['location_id'].map(mapa_regiones)
df_visitas['nombre_tienda'] = df_visitas['location_id'].map(mapa_nombres)

df_master = pd.merge(
    df_visitas, df_clima, 
    left_on=['fecha', 'location_id'], right_on=['fecha', 'location_uuid'], how='left'
)
df_master['llueve'] = df_master['llueve'].fillna(0).astype(int)

def asignar_festivo(row):
    try:
        if pd.isna(row['region_code']): return 0
        calendario = holidays.Spain(subdiv=row['region_code'], years=row['fecha'].year)
        return 1 if row['fecha'] in calendario else 0
    except:
        return 0

df_master['es_festivo'] = df_master.apply(asignar_festivo, axis=1)
df_master['es_finde'] = df_master['fecha'].dt.dayofweek.isin([5, 6]).astype(int)

df_master['dia_semana'] = df_master['fecha'].dt.dayofweek
df_master['dia_mes'] = df_master['fecha'].dt.day
df_master['mes'] = df_master['fecha'].dt.month

In [2]:
uuid_objetivo = '67034276-0d01-4c90-a363-fa75699a19a4'
nombre_objetivo = mapa_nombres.get(uuid_objetivo, "Ubicacion Desconocida")

print("="*60)
print(f"INICIANDO LABORATORIO PARA: {nombre_objetivo}")
print(f"UUID: {uuid_objetivo}")
print("="*60)

df_tienda = df_master[df_master['location_id'] == uuid_objetivo].copy()
df_tienda = df_tienda.groupby('fecha').agg({
    'total_visits': 'sum',
    'llueve': 'max',
    'es_festivo': 'max',
    'es_finde': 'max',
    'dia_semana': 'max',
    'dia_mes': 'max',
    'mes': 'max'
}).reset_index().sort_values('fecha').reset_index(drop=True)

df_tienda['lag_1d'] = df_tienda['total_visits'].shift(1)
df_tienda['lag_7d'] = df_tienda['total_visits'].shift(7)
df_tienda['media_7d'] = df_tienda['total_visits'].rolling(7).mean()

df_tienda = df_tienda.dropna().reset_index(drop=True)

print(f"\nTotal de dias validos para entrenar: {len(df_tienda)} dias")
print("\nVISTAZO AL TENSOR DE DATOS (Primeras 5 filas):")
display(df_tienda.head())

INICIANDO LABORATORIO PARA: Malaga Muelle 1
UUID: 67034276-0d01-4c90-a363-fa75699a19a4

Total de dias validos para entrenar: 217 dias

VISTAZO AL TENSOR DE DATOS (Primeras 5 filas):


,fecha,total_visits,llueve,es_festivo,es_finde,dia_semana,dia_mes,mes,lag_1d,lag_7d,media_7d
0,2025-09-09,13374,0,0,0,1,9,9,14655.0,13906.0,15454.571429
1,2025-09-10,13578,0,0,0,2,10,9,13374.0,14065.0,15385.000000
2,2025-09-11,13905,0,0,0,3,11,9,13578.0,13817.0,15397.571429
3,2025-09-12,16443,0,0,0,4,12,9,13905.0,14638.0,15655.428571
4,2025-09-13,19792,0,0,1,5,13,9,16443.0,18185.0,15885.000000


In [3]:
from sklearn.metrics import mean_absolute_error
import numpy as np

fecha_corte_test = '2026-03-01' 

train = df_tienda[df_tienda['fecha'] < fecha_corte_test].copy()
test = df_tienda[df_tienda['fecha'] >= fecha_corte_test].copy()

print("="*60)
print("CONFIGURACION DEL PERIODO DE ANALISIS")
print("="*60)
print(f"ENTRENAMIENTO: Desde {train['fecha'].min().date()} hasta {train['fecha'].max().date()} ({len(train)} dias)")
print(f"TEST (VALIDACION): Desde {test['fecha'].min().date()} hasta {test['fecha'].max().date()} ({len(test)} dias)")
print("="*60)

features = ['es_finde', 'es_festivo', 'llueve', 'dia_semana', 'dia_mes', 'mes', 'lag_1d', 'lag_7d', 'media_7d']

X_train, y_train = train[features], train['total_visits']
X_test, y_test = test[features], test['total_visits']

eval_set = [(X_train, y_train), (X_test, y_test)]

modelo = xgb.XGBRegressor(
    n_estimators=1000,    
    max_depth=4, 
    learning_rate=0.1, 
    random_state=42,
    early_stopping_rounds=15  
)

print("Iniciando ciclo de entrenamiento con parada temprana...")
modelo.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=10 
)

test['prediccion'] = modelo.predict(X_test)
test['prediccion'] = test['prediccion'].apply(lambda x: max(0, round(x)))

mae = mean_absolute_error(test['total_visits'], test['prediccion'])
suma_errores = np.sum(np.abs(test['total_visits'] - test['prediccion']))
suma_real = np.sum(test['total_visits'])
wmape = suma_errores / suma_real if suma_real > 0 else 0
accuracy = (1 - wmape) * 100

print("="*60)
print(f"ENTRENAMIENTO CONGELADO EN EL ARBOL: {modelo.best_iteration}")
print("="*60)
print(f"RESULTADOS EN EL PERIODO DE TEST:")
print(f"MAE: {mae:.1f} visitas | Accuracy: {accuracy:.2f}%")

CONFIGURACION DEL PERIODO DE ANALISIS
ENTRENAMIENTO: Desde 2025-09-09 hasta 2026-02-28 (173 dias)
TEST (VALIDACION): Desde 2026-03-01 hasta 2026-04-13 (44 dias)
Iniciando ciclo de entrenamiento con parada temprana...
[0]	validation_0-rmse:5171.59430	validation_1-rmse:3612.21580
[10]	validation_0-rmse:2387.38392	validation_1-rmse:2468.80997
[20]	validation_0-rmse:1424.71304	validation_1-rmse:2265.64553
[30]	validation_0-rmse:1054.38216	validation_1-rmse:2254.75516
[40]	validation_0-rmse:847.39065	validation_1-rmse:2319.48931
[46]	validation_0-rmse:764.75429	validation_1-rmse:2343.74778
ENTRENAMIENTO CONGELADO EN EL ARBOL: 32
RESULTADOS EN EL PERIODO DE TEST:
MAE: 1739.6 visitas | Accuracy: 86.44%


In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test['fecha'], 
    y=test['total_visits'],
    mode='lines+markers',
    name='Realidad (Visitantes)',
    line=dict(color='black', width=2),
    marker=dict(size=6)
))

fig.add_trace(go.Scatter(
    x=test['fecha'], 
    y=test['prediccion'],
    mode='lines+markers',
    name='Modelo XGBoost (Prediccion)',
    line=dict(color='#8e44ad', width=2, dash='dot'),
    marker=dict(size=6, symbol='cross')
))

for index, row in test.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['fecha'], row['fecha']],
        y=[row['total_visits'], row['prediccion']],
        mode='lines',
        line=dict(color='red', width=1),
        showlegend=False,
        hoverinfo='skip'
    ))

fig.update_layout(
    title=f'Inferencia vs Realidad en {nombre_objetivo} (Test Set)',
    xaxis_title='Fecha',
    yaxis_title='Visitantes Totales',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

In [5]:

resultados = modelo.evals_result()
epochs = len(resultados['validation_0']['rmse'])
x_axis = range(0, epochs)

pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)

gradiente_train = pred_train - y_train.values
gradiente_test = pred_test - y_test.values


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.plot(x_axis, resultados['validation_0']['rmse'], label='Entrenamiento (La Fábrica)', color='#3498db', linewidth=2.5)
ax1.plot(x_axis, resultados['validation_1']['rmse'], label='Validación (Control de Calidad)', color='#e74c3c', linewidth=2.5)


ax1.axvline(modelo.best_iteration, color='black', linestyle='--', linewidth=2, 
            label=f'Punto Óptimo / Early Stop (Árbol {modelo.best_iteration})')

ax1.set_title('Evolución del Error Árbol a Árbol', fontsize=14)
ax1.set_xlabel('Iteraciones (Número de Árboles creados)')
ax1.set_ylabel('Error (RMSE)')
ax1.legend()
ax1.grid(True, alpha=0.3)

sns.kdeplot(gradiente_train, ax=ax2, label='Gradiente en Entrenamiento', color='#3498db', fill=True, alpha=0.4)
sns.kdeplot(gradiente_test, ax=ax2, label='Gradiente en Validación', color='#e74c3c', fill=True, alpha=0.4)

ax2.axvline(0, color='black', linestyle='--', linewidth=2, label='Cero Perfecto (0 Error)')
ax2.set_title('Distribución del Gradiente Final (Realidad vs Predicción)', fontsize=14)
ax2.set_xlabel('Diferencia en Visitantes (Predicción - Realidad)')
ax2.set_ylabel('Densidad de Días')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'subplots'